In [2]:
from sklearn.datasets import load_iris
import pandas as pd

iris_data = load_iris()

df = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
df["target"] = iris_data.target

from sklearn.model_selection import train_test_split

x = df.drop(columns="target")
y = df["target"]

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=40, test_size=.20, stratify=y)

In [2]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(x_train, y_train)
forest = RandomForestClassifier(random_state=42).fit(x_train, y_train)

print("Tree  :", accuracy_score(y_test, tree.predict(x_test)))
print("Forest:", accuracy_score(y_test, forest.predict(x_test)))

print(export_text(tree, feature_names=list(x_train.columns)))   # readable rules

Tree  : 0.9666666666666667
Forest: 0.9333333333333333
|--- petal length (cm) <= 2.45
|   |--- class: 0
|--- petal length (cm) >  2.45
|   |--- petal width (cm) <= 1.55
|   |   |--- petal length (cm) <= 5.00
|   |   |   |--- class: 1
|   |   |--- petal length (cm) >  5.00
|   |   |   |--- class: 2
|   |--- petal width (cm) >  1.55
|   |   |--- petal width (cm) <= 1.75
|   |   |   |--- class: 1
|   |   |--- petal width (cm) >  1.75
|   |   |   |--- class: 2



In [3]:
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

models = {
    "lr":  make_pipeline(StandardScaler(), LogisticRegression()),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "nb":  GaussianNB(),
}

preds, probs = [], []
for name, m in models.items():
    m.fit(x_train, y_train)               # each trains independently
    preds.append(m.predict(x_test))       # label from each model
    probs.append(m.predict_proba(x_test)) # probabilities from each model

# Hard vote: most common label across the 3 models
hard = stats.mode(np.array(preds), axis=0, keepdims=False).mode

# Soft vote: average the probabilities, then pick the highest
soft = np.mean(probs, axis=0).argmax(axis=1)   # works when labels are 0, 1, 2...

print("Hard vote:", accuracy_score(y_test, hard))
print("Soft vote:", accuracy_score(y_test, soft))

Hard vote: 0.9333333333333333
Soft vote: 0.9333333333333333


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

reg_models = [
    LinearRegression(),
    make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    DecisionTreeRegressor(max_depth=4, random_state=42),
]

all_preds = [m.fit(x_train, y_train).predict(x_test) for m in reg_models]
average = np.mean(all_preds, axis=0)

print("R2:", r2_score(y_test, average))

R2: 0.9399693830997542


In [5]:
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(x_train, y_train)
print(gb.score(x_test, y_test))

0.9666666666666667


In [ ]:
import time
from xgboost import XGBClassifier

print(x_train.shape)                     # how big is the data?
print(y_train.nunique())                 # how many classes?

t = time.time()
XGBClassifier(n_estimators=400, max_depth=6, random_state=42).fit(x_train, y_train)
print("one slowest fit:", round(time.time() - t, 1), "seconds")

(120, 4)
3


In [7]:
import pandas as pd
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

log = []

def record(name, params, model, average="macro"):
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    score = f1_score(y_test, preds, average=average)
    log.append({"model": name, **params, f"f1_{average}": score})

record("XGBoost", {"lr": 0.05, "depth": 4},
       XGBClassifier(learning_rate=0.05, max_depth=4, random_state=42))

pd.DataFrame(log).to_csv("experiments.csv", index=False)

In [3]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 6],
    "subsample": [0.7, 0.8, 1.0],
}

search = RandomizedSearchCV(
    XGBClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42
)
search.fit(x_train, y_train)

print(search.best_params_)
print(search.best_score_)

KeyboardInterrupt: 

In [5]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression()),
    "Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, m in models.items():
    scores = cross_val_score(m, x_train, y_train, cv=5, scoring="f1")
    rows.append({"model": name, "mean_f1": scores.mean(), "std": scores.std()})

print(pd.DataFrame(rows).sort_values("mean_f1", ascending=False))

/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 174, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 469, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/abijithpersonal/Documents/Recap_

      model  mean_f1  std
0  Logistic      NaN  NaN
1    Forest      NaN  NaN
2   XGBoost      NaN  NaN


/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 174, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 469, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/abijithpersonal/Documents/Recap_ML/.venv/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/abijithpersonal/Documents/Recap_